# Bundle Authoring Guide Notebook

Mirrors the [Bundle Authoring Guide](https://gongjr0.github.io/SymbolicDSGE/latest/guides/bundle_authoring_guide/). Each section follows the same headings as the guide.

Run from `docs/assets/` so the relative path to `MODELS/POST82.yaml` resolves.

In [1]:
import io

import numpy as np
import pandas as pd

from SymbolicDSGE import BundleBuilder, DSGESolver, Estimator, ModelParser, Shock
from SymbolicDSGE.bayesian import make_prior

from SymbolicDSGE.monte_carlo import MCPipeline
from SymbolicDSGE.monte_carlo.step_factories import (
    jarque_bera_test_step,
    simulation_step,
)

## Solve a reference model

In [2]:
# Create the Bundle
bundle = BundleBuilder(created_by="experiment-1")

parser = ModelParser("../../MODELS/POST82.yaml")
model, kalman = parser.get_all()

solver = DSGESolver(model, kalman)
compiled = solver.compile(linearize=False)
sol = solver.solve(
    compiled,
    ss_seed=[0.0, 0.0, 0.0, 0.0, 0.0],
)

# Add the model to the bundle
bundle.add_model(
    "reference",
    model.source_yaml,
    compile_kwargs={"linearize": False},
    solve_kwargs={"ss_seed": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

# The `dgp` passed to `mc_pipeline.run(...)` is a runtime-only argument; it is
# NOT persisted by `add_mc`. To make `loaded.dgp` resolve, the DGP model must be
# bundled explicitly under role "dgp". Here the experiment uses the same model as
# its DGP, so we ship the same YAML (in a misspecification study this would be a
# different model's source).
bundle.add_model(
    "dgp",
    model.source_yaml,
    compile_kwargs={"linearize": False},
    solve_kwargs={"ss_seed": [0.0, 0.0, 0.0, 0.0, 0.0]},
)

## Specify the estimation tab

This section defines a small MCMC run against synthetic observed data, then stores the live `Estimator` and `MCMCResult` in the bundle.

In [3]:
priors = {
    "psi_pi": make_prior(
        distribution="normal",
        parameters={"mean": 1.5, "std": 0.25},
        transform="identity",
    ),
    "psi_x": make_prior(
        distribution="normal",
        parameters={"mean": 0.5, "std": 0.2},
        transform="identity",
    ),
}

rng = np.random.default_rng(0)
observed = rng.standard_normal((40, 3))
estim = Estimator(
    compiled=compiled,
    observables=["OutGap", "Infl", "Rate"],
    y=observed,
    priors=priors,
)
res = estim.mcmc(n_draws=1000, burn_in=200, thin=2, random_state=0)

MCMC sampling concluded in 0.09 seconds with 23672.66 iterations per second.
[Estimator:mcmc] BK stability warnings encountered during search: 117


In [4]:
# Add the estimation to the bundle with results
bundle.add_estimation(
    source=estim,
    result=res,
)

## Build a Monte Carlo pipeline

In [5]:
# Pass the `Shock` *instances* (not `.shock_generator()`): the pipeline clones
# them per replication with seeded offsets, and they serialize into the bundle.
gz_shock = Shock(seed=0, multivar=True, dist="norm")
r_shock = Shock(seed=1, multivar=False, dist="t", dist_kwargs={"df": 3})

mc_pipeline = MCPipeline(
    [
        simulation_step(
            "datagen",
            target="dgp",
            T=200,
            shocks={"e_g,e_z": gz_shock, "e_r": r_shock},
        ),
        jarque_bera_test_step(
            "jb_test", source="datagen", field="observables", column=0
        ),
    ]
)
mc_res = mc_pipeline.run(
    reference=sol,
    dgp=sol,
    n_rep=1000,
    n_jobs=-1,
    verbosity=2,
)

MC run concluded successfully in 0.03s with 35631.82 it/s.
Per-step Report:

	datagen: 0 failures, 82971.03 worker it/s (0.01 worker-s), 35631.82 wall it/s.
	jb_test: 0 failures, 1338329.79 worker it/s (0.00 worker-s), 35631.82 wall it/s.


In [6]:
# Add the Monte Carlo pipeline to the bundle
bundle.add_mc(
    pipeline=mc_pipeline,
    result=mc_res,
)

## Specify a simulation prefill

In [7]:
# Add the simulation prefill to the bundle, keyed by role
bundle.set_simulation(
    "reference",
    T=200,
    observables=True,
    shock_scale=1.0,
    shocks={
        "e_r": Shock(seed=42, dist="norm", dist_kwargs={"loc": 0.0}),
    },
)

## Add raw data

In [8]:
# Make text data
aux = pd.DataFrame(
    {
        "date": pd.date_range("2000-01-01", periods=40, freq="QS"),
        "gdp_growth": rng.standard_normal(40),
    }
)
csv_buf = io.StringIO()
aux.to_csv(csv_buf, index=False)

bundle.add_raw_data(
    name="auxiliary_series",
    data=csv_buf.getvalue(),
)

## Write the bundle to disk

In [9]:
bundle.write("experiment-1.sdsge")

WindowsPath('experiment-1.sdsge')

## Inspect the result

In [10]:
!unzip -l experiment-1.sdsge

Archive:  experiment-1.sdsge
  Length      Date    Time    Name
---------  ---------- -----   ----
     3532  28-08-2026 12:30   manifest.json
     2134  28-08-2026 12:30   model/reference.yaml
     2134  28-08-2026 12:30   model/dgp.yaml
      629  28-08-2026 12:30   estimation/spec.json
     1939  28-08-2026 12:30   estimation/observed.parquet
    18972  28-08-2026 12:30   estimation/posterior.parquet
      585  28-08-2026 12:30   estimation/result.json
     1018  28-08-2026 12:30   montecarlo/pipeline.json
      565  28-08-2026 12:30   montecarlo/result/meta.json
      176  28-08-2026 12:30   montecarlo/result/tests/test_steps.json
    11413  28-08-2026 12:30   montecarlo/result/tests/test_traces.parquet
     1282  28-08-2026 12:30   data/auxiliary_series.parquet
---------                     -------
    44379                     12 files


In [11]:
# File size of the bundle in KB
import os

sizeof = round(os.path.getsize("experiment-1.sdsge") / 1024)
print(f"Bundle size: {sizeof} KB")

Bundle size: 38 KB
